# Mean-shift clustering — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def pairwise_sq(A,B):
    A=np.asarray(A,float); B=np.asarray(B,float)
    return ((A[:,None,:]-B[None,:,:])**2).sum(axis=2)
def softmax(z):
    z=np.asarray(z,float); z=z-np.max(z)
    e=np.exp(z); return e/e.sum()
def entropy(p):
    p=np.asarray(p,float); p=p[p>0]
    return float(-(p*np.log(p)).sum())
def center(K):
    n=K.shape[0]; H=np.eye(n)-np.ones((n,n))/n
    return H@K@H
print('setup ok')

## One focused concept: the core computation inside Mean-shift clustering
These five cells isolate the base calculation, a knob turn, a contrast, an edge case, and a tiny end-to-end pass for Mean-shift clustering.

In [ ]:
# 1) Gaussian neighborhood weights around a query
x=np.array([-2.,-1.,0.,1.,2.]); q=0.; h=1.0
w=np.exp(-0.5*((x-q)/h)**2); p=w/w.sum()
plt.figure(figsize=(5,3)); plt.bar(x,p,width=.45); plt.title('Mean-shift clustering: normalized kernel weights')
assert abs(p.sum()-1)<1e-12 and p[2]==p.max()

In [ ]:
# 2) bandwidth/perplexity-style knob: larger h spreads influence
x=np.array([-2.,-1.,0.,1.,2.]); hs=np.linspace(.35,2.5,60)
ents=[entropy(np.exp(-0.5*(x/h)**2)/np.exp(-0.5*(x/h)**2).sum()) for h in hs]
plt.figure(figsize=(5,3)); plt.plot(hs,ents); plt.xlabel('scale h'); plt.ylabel('entropy'); plt.title('Mean-shift clustering: scale controls locality')
assert ents[-1]>ents[0]

In [ ]:
# 3) contrast local and broad kernels on the same points
x=np.array([-2.,-1.,0.,1.,2.])
plt.figure(figsize=(5,3))
for h in [.5,1.5]:
    p=np.exp(-0.5*(x/h)**2); p=p/p.sum(); plt.plot(x,p,'-o',label=f'h={h}')
plt.legend(); plt.title('Mean-shift clustering: local vs broad influence')
assert True

In [ ]:
# 4) symmetric pairwise affinities form a heatmap
X=np.array([[-1.],[0.],[.2],[2.]])
K=np.exp(-pairwise_sq(X,X)/(2*.7**2))
plt.figure(figsize=(4,4)); plt.imshow(K,cmap='viridis'); plt.colorbar(); plt.title('Mean-shift clustering: affinity matrix')
assert np.allclose(K,K.T) and np.allclose(np.diag(K),1)

In [ ]:
# 5) weighted average moves toward denser nearby points
x=np.array([-2.,-1.,0.,.2,2.]); q=.1; h=.8
w=np.exp(-0.5*((x-q)/h)**2); new=(w*x).sum()/w.sum()
plt.figure(figsize=(5,3)); plt.scatter(x,np.zeros_like(x),s=80); plt.axvline(q,c='k',ls='--',label='start'); plt.axvline(new,c='r',label='weighted mean'); plt.legend(); plt.title('Mean-shift clustering: one local update')
assert -0.2<new<0.2